# llminfer Advanced Features (Colab)

Covers constrained decoding, speculative decoding setup, benchmark artifacts, and backend compare.

## 0) Enable GPU runtime
In Colab: **Runtime -> Change runtime type -> GPU**

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), 'Torch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required.')

## 1) Clone + install

In [ ]:
REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'  # TODO
TARGET_DIR = '/content/llminfer'

import os, shutil
if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone {REPO_URL} {TARGET_DIR}
%cd /content/llminfer
!pip -q install -U pip
!pip -q install -e .

## 2) Constrained decoding example

In [ ]:
from llminfer import InferenceEngine, EngineConfig

MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
engine = InferenceEngine(EngineConfig(model_name=MODEL))
engine.load()

res = engine.run(
    'Explain GPUs for deep learning in 4 short bullet points.',
    max_new_tokens=128,
    temperature=0.2,
    no_repeat_ngram_size=3,
    bad_words=['joking', 'not sure'],
    stop_sequences=['\n\n'],
    seed=42,
)
print(res.generated_text)
print(res.stats)

## 3) Speculative decoding setup (optional)
Set a smaller assistant model to accelerate decode on some workloads.

In [ ]:
# from llminfer import InferenceEngine, EngineConfig
# cfg = EngineConfig(
#     model_name='Qwen/Qwen2.5-1.5B-Instruct',
#     assistant_model_name='Qwen/Qwen2.5-0.5B-Instruct',
# )
# speculative_engine = InferenceEngine(cfg)
# speculative_engine.load()
# out = speculative_engine.run('What is speculative decoding?', max_new_tokens=96, temperature=0.2)
# print(out.generated_text)
# speculative_engine.unload()

## 4) Benchmark + artifacts (JSON/CSV/plot)

In [ ]:
!llminfer bench --model Qwen/Qwen2.5-1.5B-Instruct --batch-sizes 1,2,4 --runs 3 --max-tokens 64 --plot /content/bench.png --plot-suite-dir /content/bench_plots --artifacts-dir /content/bench_artifacts

In [ ]:
!ls -la /content/bench_artifacts
from IPython.display import Image, display
display(Image('/content/bench.png'))

## 5) Compare eager vs compiled with exported artifacts

In [ ]:
!llminfer compare --model facebook/opt-125m --backends eager,compiled --batch-sizes 1,2,4 --runs 3 --plot /content/compare.png --plot-suite-dir /content/compare_plots --artifacts-dir /content/compare_artifacts

In [ ]:
import os
from IPython.display import Image, display
print(os.listdir('/content/compare_artifacts'))
if os.path.exists('/content/compare.png'):
    display(Image('/content/compare.png'))

## 6) Optional adapter usage (PEFT)
Install extra and provide adapter path/repo id.

In [ ]:
# !pip install -q peft
# ADAPTER = '<adapter-path-or-repo>'
# engine.load_adapter(ADAPTER, adapter_name='demo')
# print('Adapters:', engine.list_adapters())
# print(engine.run('Explain LoRA in one paragraph.', max_new_tokens=96, temperature=0.2).generated_text)
# engine.unload_adapter('demo')

In [ ]:
engine.unload()